In [6]:
import os
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime


# Define your MinIO credentials
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
# If running MinIO locally, use the local IP or 'localhost'. 
# If inside Docker, use 'host.docker.internal' or the container name.
MINIO_ENDPOINT = "http://127.0.0.1:9000"

MINIO_CATALOG_NAME = "minio_ice"
MINIO_BUCKET_URI = "s3a://icewh/warehouse"


LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

# STOP existing Spark session (safe to run even if not present)
try:
    spark.stop()
    print("Stopped existing Spark session.")
except Exception as e:
    print("No active Spark session or error stopping:", e)

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{MINIO_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{MINIO_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{MINIO_CATALOG_NAME}.warehouse", MINIO_BUCKET_URI) \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3")\
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000") \
    .config("spark.hadoop.hive.metastore.hbase.aggr.stats.memory.ttl", "60000")\
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


Stopped existing Spark session.

Namespace: dimension
+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|dimension|payment_method  |false      |
|dimension|supplier        |false      |
|dimension|city            |false      |
|dimension|stock_item      |false      |
|dimension|customer        |false      |
|dimension|date            |false      |
|dimension|transaction_type|false      |
|dimension|employee        |false      |
+---------+----------------+-----------+


Namespace: fact
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|fact     |purchase     |false      |
|fact     |stock_holding|false      |
|fact     |order        |false      |
|fact     |movement     |false      |
|fact     |sale         |false      |
|fact     |transaction  |false      |
+---------+-------------+-----------+


Namespace: integration
+-----------+------------------

In [ ]:
import re
matches = []
for k,v in spark.sparkContext.getConf().getAll():
    if re.search(r"\d+s\b", str(v)) or re.search(r"\d+s\b", k):
        matches.append((k,v))
if not matches:
    print("No SparkConf keys/values containing digits+s found.")
else:
    for k,v in matches:
        print(k, "=", v)


In [ ]:
# spark.catalog.setCurrentCatalog("Sales")
df_Sales_SalesPerson = spark.table("local.Sales.SalesOrderDetail").alias("sod")

df_Sales_SalesPerson = df_Sales_SalesPerson\
    .withColumn("SalesOrderID", sf.coalesce(sf.col("SalesOrderID").try_cast(sdt.IntegerType()), sf.lit(0)))\
    .withColumn("SalesOrderDetailID", sf.col("SalesOrderDetailID").try_cast(sdt.IntegerType()))\
    .withColumn("CarrierTrackingNumber", sf.col("CarrierTrackingNumber").try_cast(sdt.StringType()))\
    .withColumn("OrderQty", sf.col("OrderQty").try_cast(sdt.IntegerType()))\
    .withColumn("ProductID", sf.col("ProductID").try_cast(sdt.IntegerType()))\
    .withColumn("SpecialOfferID", sf.col("SpecialOfferID").try_cast(sdt.IntegerType()))\
    .withColumn("UnitPrice", sf.coalesce(sf.col("UnitPrice").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("UnitPriceDiscount", sf.coalesce(sf.col("UnitPriceDiscount").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("LineTotal", sf.coalesce(sf.col("LineTotal").try_cast(sdt.DecimalType(10, 2)),sf.lit(0).cast(sdt.DecimalType(10, 2))))\
    .withColumn("ModifiedDate", sf.current_date())

df_Sales_SalesPerson.printSchema()

df_Sales_SalesPerson = df_Sales_SalesPerson.drop("rowguid")

df_Sales_SalesPerson.show(5)

In [9]:
import csv
import os
import pyodbc # or your preferred DB connector

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=AshishPC;"
    "DATABASE=WideWorldImporters;"
    "UID=sa;"
    "PWD=sa123#;"
)
cnxn = pyodbc.connect(connection_string)
cursor = cnxn.cursor()

In [3]:
df = spark.read.csv("s3a://basetables/Sales_Invoices.csv", sep="~", header=True)
df.show(5)

+---------+----------+----------------+-------+----------------+---------------+----------------+-------------------+----------------+-----------+---------------------------+------------+----------------+--------+--------------------+----------------+-------------+-----------------+-----------+-----------+--------------------+---------------------+-------------------+------------+-------------------+
|InvoiceID|CustomerID|BillToCustomerID|OrderID|DeliveryMethodID|ContactPersonID|AccountsPersonID|SalespersonPersonID|PackedByPersonID|InvoiceDate|CustomerPurchaseOrderNumber|IsCreditNote|CreditNoteReason|Comments|DeliveryInstructions|InternalComments|TotalDryItems|TotalChillerItems|DeliveryRun|RunPosition|ReturnedDeliveryData|ConfirmedDeliveryTime|ConfirmedReceivedBy|LastEditedBy|     LastEditedWhen|
+---------+----------+----------------+-------+----------------+---------------+----------------+-------------------+----------------+-----------+---------------------------+------------+-----

In [10]:
import os
import csv
from pyspark.sql import SparkSession # Ensure you have a SparkSession object named 'spark'

# --- Configuration (Set this up near your SparkSession creation) ---
OUTPUT_DIR = "s3a://rawdata/tables"  # Added a 'tables' prefix for structure
DELIMITER = "~"
JDBC_DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" # Example for MSSQL
JDBC_URL = "jdbc:sqlserver://AshishPC:1433;databaseName=WideWorldImporters;encrypt=false"
USER = "sa"
PASSWORD = "sa123#"

# --- Export Loop using Spark ---

# 1. Get the list of all base tables (keep this logic)
print("Fetching list of all base tables...")
cursor.execute(
    "SELECT TABLE_SCHEMA, TABLE_NAME "
    "FROM information_schema.tables "
    "WHERE table_type = 'BASE TABLE'"
)
tables = cursor.fetchall()
print(f"Found {len(tables)} tables to export.")

# 2. Loop through each table and export its data using Spark
for schema_name, table_name in tables:
    full_table_name = f"[{schema_name}].[{table_name}]"
    
    # Define the output *folder* in MinIO
    # Spark writes data as a folder of partitioned files, not a single file.
    # output_path = os.path.join(OUTPUT_DIR, f"{schema_name}_{table_name}")

    # Use forward slash to ensure S3/Linux compatibility
    output_path = f"{OUTPUT_DIR}/{schema_name}_{table_name}"
    
    print(f"\nExporting data from: {full_table_name} to {output_path}...")
    
    try:
        # A. Read the data directly into a Spark DataFrame
        df = spark.read \
            .format("jdbc") \
            .option("url", JDBC_URL) \
            .option("dbtable", full_table_name) \
            .option("user", USER) \
            .option("password", PASSWORD) \
            .option("driver", JDBC_DRIVER) \
            .load()

        # B. Write the DataFrame directly to the S3A path (MinIO)
        df.write \
            .mode("overwrite") \
            .option("header", "true") \
            .option("sep", DELIMITER) \
            .csv(output_path) 
            
        print(f"Successfully exported table to {output_path}")

    except Exception as e:
        print(f"!!! FAILED to export table {full_table_name} via Spark. Error: {e}")

Fetching list of all base tables...
Found 48 tables to export.

Exporting data from: [Warehouse].[Colors] to s3a://rawdata/tables/Warehouse_Colors...
Successfully exported table to s3a://rawdata/tables/Warehouse_Colors

Exporting data from: [Warehouse].[Colors_Archive] to s3a://rawdata/tables/Warehouse_Colors_Archive...
Successfully exported table to s3a://rawdata/tables/Warehouse_Colors_Archive

Exporting data from: [Sales].[OrderLines] to s3a://rawdata/tables/Sales_OrderLines...
Successfully exported table to s3a://rawdata/tables/Sales_OrderLines

Exporting data from: [Warehouse].[PackageTypes] to s3a://rawdata/tables/Warehouse_PackageTypes...
Successfully exported table to s3a://rawdata/tables/Warehouse_PackageTypes

Exporting data from: [Warehouse].[PackageTypes_Archive] to s3a://rawdata/tables/Warehouse_PackageTypes_Archive...
Successfully exported table to s3a://rawdata/tables/Warehouse_PackageTypes_Archive

Exporting data from: [Warehouse].[StockGroups] to s3a://rawdata/tables/W

In [7]:
# Quick sanity checks
print("Spark version:", spark.version)
spark.sql("SHOW CATALOGS").show(truncate=False)

# Example: create an Iceberg namespace + table and a Delta path-based table
# spark.sql("CREATE NAMESPACE IF NOT EXISTS staging")
# spark.sql("CREATE NAMESPACE IF NOT EXISTS reporting")

# Create Iceberg table in reporting catalog
spark.sql(f"""
          CREATE TABLE delta_catalog.SalesOrderDetail_new(
          	SalesOrderID int,
			SalesOrderDetailID int,
			CarrierTrackingNumber string,
			OrderQty int,
			ProductID int,
			SpecialOfferID int,
			UnitPrice decimal(10,2),
			UnitPriceDiscount decimal(10,2),
			LineTotal  decimal(10,2),
			ModifiedDate DATE
			) USING delta
          PARTITIONED BY (ModifiedDate)
          LOCATION '{MINIO_BUCKET_URI}/SalesOrderDetail_new'
		  """)


Spark version: 4.0.1
+-------------+
|catalog      |
+-------------+
|reporting    |
|spark_catalog|
+-------------+



IllegalArgumentException: Cannot set a custom location for a path-based table. Expected file:////data/data_files/iceberg/WideWorldImportersDW/delta_catalog/SalesOrderDetail_new but got s3a://icewh/warehouse/SalesOrderDetail_new

In [4]:
namespaces_df = spark.sql(f"SHOW NAMESPACES IN {WH_CATALOG_NAME}")
namespaces = [row['namespace'] for row in namespaces_df.collect()]

for ns in namespaces:
    print(f"--- Processing Namespace: {ns} ---")
    
    # 2. Create the namespace in the destination (MinIO) if it doesn't exist
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {MINIO_CATALOG_NAME}.{ns}")
    
    # 3. List tables in this namespace
    tables_df = spark.sql(f"SHOW TABLES IN {WH_CATALOG_NAME}.{ns}")
    tables = [row['tableName'] for row in tables_df.collect()]
    
    for tbl in tables:
        src_table = f"{WH_CATALOG_NAME}.{ns}.{tbl}"
        dest_table = f"{MINIO_CATALOG_NAME}.{ns}.{tbl}"
        
        print(f"Migrating: {src_table}  ->  {dest_table} ...")
        
        # 4. Read from Source
        df = spark.read.table(src_table)
        
        # 5. Write to Destination (CTAS - Create Table As Select)
        # using writeTo().create() ensures Iceberg metadata is generated correctly on S3
        try:
            df.writeTo(dest_table).create()
            print(f"SUCCESS: {dest_table} created.")
        except Exception as e:
            if "Table already exists" in str(e):
                print(f"SKIPPING: {dest_table} already exists.")
            else:
                print(f"ERROR migrating {dest_table}: {e}")

print("\n--- Migration Complete ---")

# Verification
print("\nVerifying tables in MinIO:")
spark.sql(f"SHOW NAMESPACES IN {MINIO_CATALOG_NAME}").show()

--- Processing Namespace: dimension ---
Migrating: reporting.dimension.payment_method  ->  minio_ice.dimension.payment_method ...
SUCCESS: minio_ice.dimension.payment_method created.
Migrating: reporting.dimension.supplier  ->  minio_ice.dimension.supplier ...
SUCCESS: minio_ice.dimension.supplier created.
Migrating: reporting.dimension.city  ->  minio_ice.dimension.city ...
SUCCESS: minio_ice.dimension.city created.
Migrating: reporting.dimension.stock_item  ->  minio_ice.dimension.stock_item ...
SUCCESS: minio_ice.dimension.stock_item created.
Migrating: reporting.dimension.customer  ->  minio_ice.dimension.customer ...
SUCCESS: minio_ice.dimension.customer created.
Migrating: reporting.dimension.date  ->  minio_ice.dimension.date ...
SUCCESS: minio_ice.dimension.date created.
Migrating: reporting.dimension.transaction_type  ->  minio_ice.dimension.transaction_type ...
SUCCESS: minio_ice.dimension.transaction_type created.
Migrating: reporting.dimension.employee  ->  minio_ice.dimens